In [ ]:
from pathlib import Path
import yaml

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import shufflenet_v2_x1_0, ShuffleNet_V2_X1_0_Weights  
import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib.pyplot as plt

from tqdm import tqdm

# Moduli progetto
from models.model import ChimeraSeg
from models.decoder import Decoder
from training.train import Trainer
from training.metrics import MeanIoU
from data.streethazards import StreetHazards
from utils.misc import get_device, fix_random, print_summary

# Configurazioni di ambiente
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload



Premessa: come il mio precedente notebook, link al notebook, sono interessato ad avere delle reti efficienti che mi permettano di poter trainare liberamente e fare diverse prove anche su un dispositivo mac con `mps`.

Guardando i diversi paper che ottenevano alti score nel benchmark SegmentMeIfYouCan, i migliori come mIoU utilizzano Mask classification e di conseguenza Mask2Former con l'aggiunta di un modulo o di tecniche per aggiungere anche il rilevamento di anomalie. Tra tutti ho preferito implementare il paper `Open Semantic Segmentation with Class Similarity` che ottiene un mIoU competitivo, rimanendo al contempo veloce a differenza degli approcci con Mask2Former più lenti di natura, e con un rilevamento di anomalie decisamente migliore. Successivamente, data la recente uscita del paper `Open Panoptic Segmentation`, sempre degli stessi autori, che oltre ad aggiungere la Panoptic Segmentation, risolve alcuni problemi del precedente approccio.

In [15]:
config_path = "./config.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

fix_random(config['seed'])
device = get_device()

🚀 CUDA device is available!


In [16]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision import transforms

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

data_transforms = {
    "train": A.Compose([
        A.Resize(height=224, width=224),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.4),
        A.OneOf([
            A.RandomRain(p=1),
            A.RandomFog(p=1),
            A.RandomShadow(p=1),
            A.RandomSunFlare(p=1)
        ], p=0.4),
        A.MotionBlur(blur_limit=7, p=0.3),
        A.Normalize(mean=mean, std=std, max_pixel_value=255.0),
        ToTensorV2(),
    ]),
    "val": A.Compose([
        A.Resize(height=224, width=224),
        A.Normalize(mean=mean, std=std, max_pixel_value=255.0),
        ToTensorV2(),
    ])
}

denorm = transforms.Normalize(
    mean=[-m/s for m, s in zip(mean, std)],
    std=[1/s for s in std]
)

In [ ]:
train_root = Path.home() / config['paths']['train_path']

train_data = StreetHazards(
    train_root,
    "training",
    transforms=data_transforms["train"]
)

val_data = StreetHazards(
    train_root,
    "validation",
    transforms=data_transforms["val"]
)

num_classes = len(.CLASSES)
class_dict = train_data.get_classes_as_dict()

print(f"Number of training samples: {len(train_data)}")
print(f"Number of validation samples: {len(val_data)}")
print(f"Number of classes: {num_classes}")

AttributeError: 'StreetHazards' object has no attribute 'CLASSES'

In [18]:
from torchvision.utils import make_grid
import numpy as np


def visualize_augmentations(dataset, num_samples=4):
    img, _ = dataset[0]

    imgs = []

    for _ in range(num_samples - 1):
        img_np = img.permute(1, 2, 0).numpy()
        augmented = train_transform(image=img_np)['image']
        denormalized = denorm(augmented)
        imgs.append(denormalized)

    img_grid = make_grid(imgs, nrow=4, normalize=True)

    plt.figure(figsize=(15,15))
    plt.imshow(img_grid.permute(1, 2, 0).numpy())
    plt.title("Original and Augmented Images", fontsize=16)
    plt.axis('off')
    plt.show()

visualize_augmentations(train_data)

NameError: name 'train_transform' is not defined

In [19]:
class MultiEpochsDataLoader(torch.utils.data.DataLoader):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._DataLoader__initialized = False
        self.batch_sampler = _RepeatSampler(self.batch_sampler)
        self._DataLoader__initialized = True
        self.iterator = super().__iter__()

    def __len__(self):
        return len(self.batch_sampler.sampler)

    def __iter__(self):
        for i in range(len(self)):
            yield next(self.iterator)


class _RepeatSampler(object):
    """ Sampler that repeats forever.

    Args:
        sampler (Sampler)
    """

    def __init__(self, sampler):
        self.sampler = sampler

    def __iter__(self):
        while True:
            yield from iter(self.sampler)

In [20]:
import os
NUM_WORKERS = os.cpu_count() - 1

train_loader = MultiEpochsDataLoader(
    train_data,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

val_loader = MultiEpochsDataLoader(
    val_data,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")

Number of training batches: 161
Number of validation batches: 33


In [21]:
do_train = True 

pretrained_encoder = shufflenet_v2_x1_0(weights=ShuffleNet_V2_X1_0_Weights.DEFAULT)
backbone = nn.Sequential(*(list(pretrained_encoder.children())[:-2]))
decoder = Decoder(464, 224, num_classes)
model = ChimeraSeg(backbone, decoder)
model.forward(torch.rand(64, 3, 224, 224))
print_summary(model, (1, 3, 224, 224))

trainer = Trainer(
    config, 
    model, 
    device, 
    train_loader, 
    val_loader,
    class_dict=class_dict, 
    metrics=[MeanIoU(num_classes)]
)

if do_train:
    trainer.train("debug")
else:
    pass

Layer (type:depth-idx)                             Output Shape              Param #
ChimeraSeg                                         [1, 464, 7, 7]            --
├─Sequential: 1-1                                  [1, 464, 7, 7]            --
│    └─Sequential: 2-1                             [1, 24, 112, 112]         --
│    │    └─Conv2d: 3-1                            [1, 24, 112, 112]         648
│    │    └─BatchNorm2d: 3-2                       [1, 24, 112, 112]         48
│    │    └─ReLU: 3-3                              [1, 24, 112, 112]         --
│    └─MaxPool2d: 2-2                              [1, 24, 56, 56]           --
│    └─Sequential: 2-3                             [1, 116, 28, 28]          --
│    │    └─InvertedResidual: 3-4                  [1, 116, 28, 28]          7,398
│    │    └─InvertedResidual: 3-5                  [1, 116, 28, 28]          7,598
│    │    └─InvertedResidual: 3-6                  [1, 116, 28, 28]          7,598
│    │    └─InvertedResid

Epoch:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1 Batches:   0%|          | 0/161 [00:00<?, ?it/s]

Epoch 2 Batches:   0%|          | 0/161 [00:00<?, ?it/s]

Evaluating train:   0%|          | 0/161 [00:00<?, ?it/s]

...

wandb: WARNING Tried to log to step 2898 that is less than the current step 2899. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 2898 that is less than the current step 2899. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


Epoch 20 Batches:   0%|          | 0/161 [00:00<?, ?it/s]

Evaluating train:   0%|          | 0/161 [00:00<?, ?it/s]

Evaluating val:   0%|          | 0/33 [00:00<?, ?it/s]

train/MeanIoU,▁▃▅▆▇▇██
train/batch_loss,█▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch_loss,█▅▃▂▂▁▁▁
train/lr,▇▇██████▇▇▆▆▆▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁
train/MeanIoU,0.51532
train/batch_loss,0.36406
train/epoch_loss,0.27455
train/lr,0.0


In [ ]:
train_loader._shutdown_workers()
val_loader._shutdown_workers()

0

In [8]:
# from torchvision.utils import make_grid

# NUM_IMAGES = 6
# grid = make_grid([train_data[i][0] for i in range(NUM_IMAGES)], nrow=3)

# grid_image = grid.permute(1, 2, 0)

# plt.figure(figsize=(15, 5))
# plt.imshow(grid_image.numpy())
# plt.axis('off')
# plt.show()

In [48]:
# from utils.visualize import color, COLORS

# color(train_data[100][2], COLORS)

nel training devo vedere due metriche mIoU, mAP, F1 score? e 